# Study report: default-latencies

The study measures the three latencies of the kernel with the board and
kernel defaults and an idle CPU: M1 (IRQ entry), M2 (ISR to thread) and
M3 (thread to thread). It has no experiments of its own. Its three
sub-studies give the answer: `irq-entry`, `isr-to-thread` and
`thread-to-thread`. Each sub-study has its own report with the modes and
the hypotheses. This notebook is a short overview from the data that the
sub-studies read.

This notebook reads one built instance of the study (`study.py analysis
build`). `study.py analysis notebooks` runs it and writes the HTML into the
instance folder. In the figures: hover for values, drag to zoom, click a
legend entry to hide a series, and use the menu above a figure to select
the metric. Commit this file without outputs.

In [ ]:
# Parameters. The instance folder: LATBENCH_INSTANCE ("study.py analysis notebooks" sets it), else the
# folder of this notebook when it is the executed copy in <instance>/notebooks/, else the instance of this
# study in LATBENCH_RESULTS (a results root). The safety repo: LATBENCH_REPO, else the design repo of the
# lab space of the instance (.lab/config), else the repo of the current folder.
import configparser
import os
import subprocess
import sys
from pathlib import Path

here = Path.cwd()
INSTANCE = Path(os.environ.get("LATBENCH_INSTANCE") or next(
    (d for d in (here, here.parent) if (d / "study.json").is_file() and (d / "instance.json").is_file()),
    Path(os.environ.get("LATBENCH_RESULTS", "/wrk/z/lab-kernel-latency/results/kernel-latency")) / "st-default-latencies@nucleo_g474re"))
lab = next((d for d in INSTANCE.resolve().parents if (d / ".lab" / "config").is_file()), None)
if os.environ.get("LATBENCH_REPO"):
    REPO = Path(os.environ["LATBENCH_REPO"])
elif lab:
    cfg = configparser.ConfigParser()
    cfg.read(lab / ".lab" / "config")
    REPO = lab / cfg.get("design", "repo", fallback="safety")
else:
    REPO = Path(subprocess.run(["git", "rev-parse", "--show-toplevel"], capture_output=True, text=True,
                               check=True).stdout.strip())

In [ ]:
import pandas as pd

sys.path.insert(0, str(REPO / "scripts" / "probatrix" / "benchmarks"))
import data  # noqa: E402
import nbfig  # noqa: E402

pd.set_option("display.max_colwidth", None)
inst = data.load_instance(INSTANCE)
nbfig.intro(inst)

## Central results

In [ ]:
inst.central()

## Hypotheses

The study has no hypotheses of its own. The reports of the sub-studies
show their hypotheses and checks.

## The three latencies

- M1, IRQ entry: the time from the edge to the first instruction of the
  ISR (`irq_latency`, experiment `irq-baseline-1m`, metric `irq`).
- M2, ISR to thread: the time from the ISR to the first instruction of
  the thread that the ISR wakes (`wake_latency`, experiment
  `wake-isr-1m`, metric `isr2thr`).
- M3, thread to thread: the time from a wake call in one thread to the
  first instruction of the woken thread (`thread_latency`, experiment
  `thread-baseline-1m`, metric `t2t`).

The CCDF shows the point of the central result of each sub-study: the
direct path for M1, the semaphore from the wrapped ISR for M2 and the
semaphore for M3. The M3 experiment uses a stimulus interval of
50..150 us. The M1 and M2 experiments use 100..500 us.

In [ ]:
MEASURES = [("M1", "irq-baseline-1m", "irq"), ("M2", "wake-isr-1m", "isr2thr"), ("M3", "thread-baseline-1m", "t2t")]
MEASURES = [m for m in MEASURES if nbfig.built(inst, m[1])]
central = {c["id"]: c for c in inst.study["central_results"]}
series, rows = {}, []
for c in central.values():
    for r in c.get("rows", []):
        m = next((m for m in MEASURES if r["run"].startswith(m[1] + "/")), None)
        if m and r["metric"] == m[2]:
            label = f"{m[0]} {m[1]} {r['point']}"
            series[label] = {"latency": inst.hist(r["point"], m[2], m[1], tree=r["tree"])}
            df = inst.rows(m[1])
            rows.append(df[(df.point == r["point"]) & (df.metric == m[2]) & (df.tree == r["tree"])]
                        .assign(measure=m[0], experiment=m[1], central=c["id"]))
nbfig.ccdf(series, "The three latencies: the points of the central results", ["latency"]).show()
pd.concat(rows).set_index(["measure", "experiment", "point", "metric"])[
    ["central", "n", "p50", "p99", "p99.9", "p99.99", "max", "mean"]]

## All points of the three experiments

The p50, the p99 and the max of each point. The label of a point
starts with its measurement (M1, M2, M3). M1: the four interrupt paths. M2: the five
kernel objects from the wrapped ISR, and the semaphore from the direct
ISR. M3: the eleven kernel objects of `thread_latency`, and the
semaphore and the mutex without preemption.

In [ ]:
parts = []
for name, exp, metric in MEASURES:
    df = inst.rows(exp)
    parts.append(df[df.metric == metric].assign(measure=name, metric="latency"))
allp = pd.concat(parts)
allp["point"] = allp["measure"] + " " + allp["point"]
nbfig.dots(allp, "The three latencies per point", stats=("p50", "p99", "max")).show()
allp.set_index(["measure", "point"])[["p50", "p99", "p99.99", "max", "mean"]]